Mit Colab kann das Jupyter-Notebook interaktiv im Browser gestartet werden:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opendatazurich/opendatazurich.github.io/blob/master/geoportal/Geoportal_Bauten_Blockmodell.ipynb)

# 3D-Blockmodell via WFS beziehen

Das Blockmodell der Stadt Zürich enthält die Gebäudegrundrisse der Amtlichen Vermessung, erweitert um
die photogrammetrisch bestimmten Höhen **Traufe** und **First**. Damit lässt sich jedes Gebäude als
Block darstellen — daher der Name.

Dieses Notebook zeigt, wie diese Daten über einen **WFS** (Web Feature Service) abgefragt, als
GeoJSON gespeichert, in GeoPandas tabellarisch ausgewertet und auf einer Karte dargestellt werden.

Der Datensatz heisst im OGD-Katalog
[Bauten - Blockmodell (inkl. 3D)](https://data.stadt-zuerich.ch/dataset/geo_bauten___blockmodell);
dort ist auch die WFS-URL aufgeführt. Frühere Versionen dieses Notebooks verwendeten den Dienst
`3D_Blockmodell_LoD1` — dieser existiert nicht mehr.

In [1]:
#%pip install --quiet geopandas requests folium

In [2]:
import xml.etree.ElementTree as ET

import geopandas
import requests
import folium

## Dienst und Version

Die WFS-URL steht im OGD-Katalog bei der Ressource «Web Feature Service». Dort ist der Dienst mit
`VERSION=1.1.0` publiziert — diese Version wird hier durchgehend verwendet.

In [3]:
wfs_url = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Bauten___Blockmodell"
WFS_VERSION = "1.1.0"

## GetCapabilities

`GetCapabilities` beschreibt, was der Server kann: welche Layer er anbietet, in welchen
Koordinatensystemen und in welchen Ausgabeformaten.

In [4]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetCapabilities',
})
print(r.status_code, r.headers['content-type'], len(r.content), "Bytes")
print(r.text[:600])

200 text/xml; charset=utf-8 6072 Bytes
<WFS_Capabilities xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ows="http://www.opengis.net/ows" xmlns:gml="http://www.opengis.net/gml" xsi:schemaLocation="http://www.opengis.net/wfs http://schemas.opengis.net/wfs/1.1.0/wfs.xsd" updateSequence="0" xmlns:ogc="http://www.opengis.net/ogc" xmlns="http://www.opengis.net/wfs" version="1.1.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
 <ows:ServiceIdentification>
  <ows:Title>Untitled</ows:Title>
  <ows:ServiceType>WFS</ows:ServiceType>
  <ows:ServiceTypeVersion>1.1.0</ows:ServiceTypeVersion>
  <ows:Fees>None</ows:Fees>
  <ows:AccessCo


### Layer auslesen

In WFS 1.1.0 heissen die Koordinatensystem-Elemente `DefaultSRS` und `OtherSRS` (in WFS 1.0.0 hiess
das Element noch `SRS`). Mit `findtext()` statt `find().text` bekommt man `None` statt eines
`AttributeError`, wenn ein Element fehlt.

In [5]:
root = ET.fromstring(r.content)
namespaces = {
    'wfs': 'http://www.opengis.net/wfs',
    'ows': 'http://www.opengis.net/ows',
}

layers = {}
for feature_type in root.findall('wfs:FeatureTypeList/wfs:FeatureType', namespaces):
    name = feature_type.findtext('wfs:Name', namespaces=namespaces)
    layers[name] = {
        'title': feature_type.findtext('wfs:Title', namespaces=namespaces),
        'default_srs': feature_type.findtext('wfs:DefaultSRS', namespaces=namespaces) or None,
        'other_srs': [e.text for e in feature_type.findall('wfs:OtherSRS', namespaces)],
    }

layers

{'bauten_blockmodell_2d': {'title': 'bauten_blockmodell_2d',
  'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']},
 'bauten_blockmodell_max_3d': {'title': 'bauten_blockmodell_max_3d',
  'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']},
 'bauten_blockmodell_min_3d': {'title': 'bauten_blockmodell_min_3d',
  'default_srs': None,
  'other_srs': ['EPSG:4326', 'EPSG:3857']}}

Der Dienst bietet drei Layer:

| Layer | Inhalt |
|---|---|
| `bauten_blockmodell_2d` | Grundrisse ohne Höhen |
| `bauten_blockmodell_max_3d` | Blöcke bis zum **First** (Dachspitze) |
| `bauten_blockmodell_min_3d` | Blöcke bis zur **Traufe** (Dachrand) |

Beachten: **`DefaultSRS` ist leer.** Das Koordinatensystem lässt sich hier also nicht aus den
Capabilities lesen, es muss später explizit gesetzt werden. GeoJSON liefert dieser Server
standardmässig in WGS 84 (EPSG:4326).

Die Attribute der Layer sind auf der
[Geoportal-Seite des Datensatzes](https://www.stadt-zuerich.ch/geodaten/download/Bauten___Blockmodell)
beschrieben.

In [6]:
layer = 'bauten_blockmodell_max_3d'

### Ausgabeformate

In WFS 1.1.0 stehen die Ausgabeformate unter `ows:OperationsMetadata` beim Parameter `outputFormat`
(in WFS 1.0.0 war es `wfs:Capability/wfs:Request/wfs:GetFeature/wfs:ResultFormat`).

In [7]:
output_formats = []
for operation in root.findall('ows:OperationsMetadata/ows:Operation', namespaces):
    if operation.get('name') != 'GetFeature':
        continue
    for parameter in operation.findall('ows:Parameter', namespaces):
        if parameter.get('name') == 'outputFormat':
            output_formats = [v.text for v in parameter.findall('ows:Value', namespaces)]

output_formats

['text/xml; subtype=gml/2.1.2',
 'text/xml; subtype=gml/3.1.1',
 'application/vnd.geo+json']

## Wie viele Daten sind das?

Vor dem Herunterladen lohnt sich ein Blick auf die Anzahl Features. Dafür gibt es
`resultType=hits`: der Server liefert dann nur die Anzahl, keine Geometrien.

In [8]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'resultType': 'hits',
})
hits = ET.fromstring(r.content)
anzahl = int(hits.get('numberOfFeatures'))
print(f"{anzahl} Features im Layer {layer}")

66830 Features im Layer bauten_blockmodell_max_3d


Das sind alle Gebäude der Stadt. Als GeoJSON wären das über 200 MB — für ein Notebook zu viel.
Deshalb wird hier nur ein Ausschnitt geholt, über den Parameter `bbox`.

Das Fenster ist in **LV95** (EPSG:2056) angegeben, im Format
`minx,miny,maxx,maxy,EPSG:2056`. Hier ein Quadrat von 300 m Seitenlänge rund um den Hauptbahnhof.

In [9]:
bbox = '2683000,1247350,2683300,1247650,EPSG:2056'

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'bbox': bbox,
})
blockmodell = r.json()
print(len(blockmodell['features']), "Features,", len(r.content), "Bytes")

291 Features, 1178499 Bytes


Die Geometrien sind echte 3D-Koordinaten: pro Punkt kommen drei Werte zurück
(Länge, Breite, Höhe über Meer).

In [10]:
geometrie = blockmodell['features'][0]['geometry']
punkt = geometrie['coordinates'][0][0][0]
print(geometrie['type'], '->', punkt)

MultiPolygon -> [8.540921, 47.37343, 419.4]


## Daten in GeoPandas einlesen

`GeoDataFrame.from_features()` macht aus dem GeoJSON eine Tabelle mit Geometriespalte.
Das Koordinatensystem muss explizit mitgegeben werden, weil es — siehe oben — nicht in den
Capabilities steht. GeoJSON kommt von diesem Server in EPSG:4326.

In [11]:
gdf = geopandas.GeoDataFrame.from_features(blockmodell, crs='EPSG:4326')
print(len(gdf), "Zeilen |", gdf.crs)
gdf.head(2)

291 Zeilen | EPSG:4326


,geometry,art,art_txt,bem,dach,egid,geometrie_gdo,gid,gid_3d,gruppierung,h_boden,h_first,h_mean,h_rel_first_boden,h_trauf,objectid,region,status,status_txt
0,"MULTIPOLYGON Z (((8.54092 47.37343 419.4, 8.54...",BB00,Gebaeude_Verwaltung,OK,NaN,302040363.0,None,z41ac39cc00001a75,z41ac39cc00001a75-00-0001,gebaeude,419.4,426.4,426.0,7.0,425.6,74,8,3,real
1,"MULTIPOLYGON Z (((8.53967 47.37219 409.8, 8.53...",BB01,Gebaeude_Wohngebaeude,OK,NaN,140749.0,None,z41ac39cc00001b45,z41ac39cc00001b45-00-0001,gebaeude,409.8,428.4,428.4,18.6,428.4,104,8,3,real


Die interessanten Attribute des Blockmodells sind die Höhen:

| Attribut | Bedeutung |
|---|---|
| `h_boden` | Höhe des Bodens (m ü. M.) |
| `h_trauf` | Höhe der Traufe, also des Dachrands |
| `h_first` | Höhe des Firsts, also der Dachspitze |
| `h_rel_first_boden` | Gebäudehöhe = First minus Boden |
| `art_txt` | Art des Bauwerks |

In [12]:
hoehen = ['h_boden', 'h_trauf', 'h_first', 'h_rel_first_boden']
gdf[hoehen].describe().round(1)

,h_boden,h_trauf,h_first,h_rel_first_boden
count,291.0,291.0,291.0,291.0
mean,410.5,426.3,429.4,19.0
std,3.9,6.8,8.2,8.6
min,404.6,410.1,410.1,0.3
25%,407.5,422.3,425.0,13.2
50%,409.5,427.9,431.1,20.1
75%,413.0,430.6,434.8,24.0
max,421.2,455.0,460.5,53.9


### LV95 statt WGS 84

Für Längen- und Flächenberechnungen braucht es ein metrisches Koordinatensystem. Zwei Wege:

1. In GeoPandas umprojizieren mit `.to_crs(epsg=2056)`.
2. Die Daten direkt in LV95 anfordern, über den Parameter `srsName=EPSG:2056`.

Der zweite Weg ist schneller, weil der Server die Umrechnung macht.

In [13]:
gdf_lv95 = gdf.to_crs(epsg=2056)
print("umprojiziert:", gdf_lv95.crs.name)

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'bbox': bbox,
    'srsName': 'EPSG:2056',
})
print("direkt vom Server:", r.json()['crs']['properties']['name'])

umprojiziert: CH1903+ / LV95


direkt vom Server: urn:ogc:def:crs:EPSG::2056


Mit der Geometrie in LV95 lässt sich zum Beispiel die Grundfläche berechnen —
die Einheit ist dann Meter, nicht Grad.

In [14]:
gdf_lv95 = gdf_lv95.assign(grundflaeche_m2=gdf_lv95.area.round(1))
gdf_lv95[['art_txt', 'h_rel_first_boden', 'grundflaeche_m2']].head()

,art_txt,h_rel_first_boden,grundflaeche_m2
0,Gebaeude_Verwaltung,7.0,130.5
1,Gebaeude_Wohngebaeude,18.6,107.3
2,Gebaeude_Wohngebaeude,20.5,534.0
3,Gebaeude_Nebengebaeude,7.3,61.8
4,Gebaeude_Wohngebaeude,23.8,306.5


## Pagination

Statt eines räumlichen Ausschnitts kann man die Features auch stückweise holen. Dazu dienen
`startIndex` (wie viele überspringen) und `maxFeatures` (wie viele holen).

In [15]:
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': WFS_VERSION,
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'startIndex': 5,
    'maxFeatures': 4,
})
seite = r.json()
[f['properties']['gid_3d'] for f in seite['features']]

['z41ac87fd00031f57-00-0001',
 'z41adf64700001d53-00-0002',
 'z41ac69e900023220-00-0001',
 'z41acbc6a000146ef-00-0001']

## GeoJSON auf die Karte bringen

Mit **folium** lassen sich die Features als GeoJSON-Layer auf eine Leaflet-Karte legen.
Leaflet ist eine 2D-Bibliothek: die Höhenwerte werden ignoriert, dargestellt werden die Grundrisse.

In [16]:
m = folium.Map(location=[47.3775, 8.5395], zoom_start=16)
folium.GeoJson(blockmodell, name=layer).add_to(m)
m

### Mit der städtischen Basiskarte als Hintergrund

Die Stadt Zürich stellt die Basiskarte als WMS bereit. Wichtig dabei: der **Layer-Name im WMS
ist `wmslayers`**, nicht der Name des Dienstes — sonst antwortet der Server mit HTTP 500 und die
Karte bleibt leer.

In [17]:
m = folium.Map(location=[47.3775, 8.5395], zoom_start=16, tiles=None)
folium.raster_layers.WmsTileLayer(
    url='https://www.ogd.stadt-zuerich.ch/wms/geoportal/Basiskarte_Zuerich_Raster_Grau',
    layers='wmslayers',
    name='Zürich - Basiskarte',
    fmt='image/png',
    overlay=False,
    control=False,
    autoZindex=False,
    attr='&copy; Stadt Zürich, Open Government Data',
).add_to(m)
folium.GeoJson(
    blockmodell,
    name='Blockmodell',
    style_function=lambda _: {'color': '#0f62fe', 'weight': 1, 'fillOpacity': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=['art_txt', 'h_rel_first_boden'],
        aliases=['Art', 'Höhe (m)'],
    ),
).add_to(m)
m